# CRISP-DM Evaluation — ENEM Opportunity & Equity Radar (2020–2024)

Six evaluation visuals from Gold layer. Same scale across years where applicable. Figures saved to `reports/figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

DIR_GOLD = ROOT / "data" / "gold"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANOS = [2020, 2021, 2022, 2023, 2024]
sns.set_style("whitegrid")

In [ ]:
spark = (
    SparkSession.builder.appName("ENEM-Evaluation-Visuals")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
fact = spark.read.parquet(str(DIR_GOLD / "fato_desempenho"))
dim_geo = spark.read.parquet(str(DIR_GOLD / "dim_geografia"))
dim_perfil = spark.read.parquet(str(DIR_GOLD / "dim_perfil"))
kpis = spark.read.parquet(str(DIR_GOLD / "kpis_uf_ano"))

## Visual 1 — Distribution of Objective Mean Score (2020–2024)

In [ ]:
# Sample per year for density (same scale 0–1000)
fact_s = fact.filter(F.col("media_objetiva").isNotNull()).sample(False, 0.05, 42)
rows = fact_s.select("ano", "media_objetiva").orderBy(F.rand()).limit(50000).collect()
by_year = {a: [] for a in ANOS}
for r in rows:
    if r.ano in by_year:
        by_year[r.ano].append(r.media_objetiva)

fig, ax = plt.subplots(figsize=(10, 5))
for ano in ANOS:
    if by_year[ano]:
        sns.kdeplot(by_year[ano], ax=ax, label=str(ano), linewidth=2)
ax.set_xlim(0, 1000)
ax.set_ylim(0, None)
ax.set_xlabel("Objective mean score (CN+CH+LC+MT)/4")
ax.set_ylabel("Density")
ax.set_title("Distribution of Objective Mean Score by Year (2020–2024)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "v1_media_objetiva_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

stats1 = {a: (np.mean(by_year[a]), np.median(by_year[a]), len(by_year[a])) for a in ANOS if by_year[a]}
print("Summary: mean, median, n per year:", stats1)

## Visual 2 — Redação Score Distribution (% >= 800)

In [ ]:
pct_800 = fact.filter(F.col("redacao").isNotNull()).groupBy("ano").agg(
    (F.sum(F.when(F.col("redacao") >= 800, 1).otherwise(0)) / F.count("*") * 100).alias("pct_800")
).orderBy("ano").collect()
years_v2 = [r.ano for r in pct_800]
pcts_v2 = [round(r.pct_800, 2) for r in pct_800]

redacao_s = fact.filter(F.col("redacao").isNotNull()).sample(False, 0.05, 42)
rows_r = redacao_s.select("ano", "redacao").limit(50000).collect()
by_year_r = {a: [] for a in ANOS}
for r in rows_r:
    if r.ano in by_year_r:
        by_year_r[r.ano].append(r.redacao)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for ano in ANOS:
    if by_year_r[ano]:
        sns.kdeplot(by_year_r[ano], ax=ax1, label=str(ano))
ax1.axvline(800, color="red", linestyle="--", label="800")
ax1.set_xlim(0, 1000)
ax1.set_ylabel("Density")
ax1.set_title("Redação Score Distribution by Year (tail >= 800 highlighted)")
ax1.legend(loc="upper left")
ax2.bar(years_v2, pcts_v2, color="steelblue", edgecolor="black")
ax2.set_xlabel("Year")
ax2.set_ylabel("% with Redação >= 800")
ax2.set_title("% of Candidates with Redação >= 800 by Year")
plt.tight_layout()
plt.savefig(FIG_DIR / "v2_redacao_800_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("% >= 800 by year:", dict(zip(years_v2, pcts_v2)))

## Visual 3 — Presence Rate by Year

In [ ]:
# Gold = valid cohort only, so presence is 100%; aggregate from kpis (weighted by count)
presence_by_year = kpis.groupBy("ano").agg(
    (F.sum(F.col("pct_presence_full") * F.col("count_participantes")) / F.sum("count_participantes")).alias("pct_presence_full")
).orderBy("ano").collect()
years_v3 = [r.ano for r in presence_by_year]
pcts_v3 = [round(r.pct_presence_full, 2) for r in presence_by_year]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(years_v3, pcts_v3, marker="o", linewidth=2, markersize=8)
ax.axvspan(2020, 2021.5, alpha=0.2, color="gray", label="Pandemic period")
ax.set_xlabel("Year")
ax.set_ylabel("% fully present")
ax.set_title("Presence Rate by Year (valid cohort = 100% in Gold)")
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "v3_presence_rate_by_year.png", dpi=150, bbox_inches="tight")
plt.show()
print("Presence % by year:", dict(zip(years_v3, pcts_v3)))

## Visual 4 — Average Score by UF (Top 10 Growth 2020 vs 2024)

In [ ]:
f_geo = fact.join(dim_geo, fact.id_geo == dim_geo.id_geo)
avg_uf_ano = f_geo.filter(F.col("sg_uf_residencia") != "NA").groupBy("sg_uf_residencia", "ano").agg(
    F.mean("media_objetiva").alias("avg_objetiva")
).filter(F.col("ano").isin(2020, 2024))
pivot = avg_uf_ano.groupBy("sg_uf_residencia").pivot("ano", [2020, 2024]).agg(F.first("avg_objetiva"))
pivot = pivot.withColumn("delta", F.col("2024") - F.col("2020")).orderBy(F.desc("delta")).limit(10)
rows_v4 = pivot.collect()

ufs = [r.sg_uf_residencia for r in rows_v4]
avg_2020 = [round(r["2020"], 1) for r in rows_v4]
avg_2024 = [round(r["2024"], 1) for r in rows_v4]
deltas = [round(r.delta, 1) for r in rows_v4]

x = np.arange(len(ufs))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, avg_2020, w, label="2020", color="steelblue")
ax.bar(x + w/2, avg_2024, w, label="2024", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(ufs)
ax.set_ylabel("Average objective mean score")
ax.set_title("Top 10 UFs by Growth in Average Score (2020 vs 2024)")
ax.legend()
ax.set_ylim(0, 1000)
for i, d in enumerate(deltas):
    ax.annotate(f"+{d}", (x[i], max(avg_2020[i], avg_2024[i]) + 15), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "v4_uf_top10_growth.png", dpi=150, bbox_inches="tight")
plt.show()
print("Top 10 UFs (delta):", list(zip(ufs, deltas)))

## Visual 5 — Gap by Income (Q006: Lowest vs Highest) Over Time

In [ ]:
f_perfil = fact.join(dim_perfil, fact.id_perfil == dim_perfil.id_perfil)
avg_renda_ano = f_perfil.filter((F.col("faixa_renda") != "NA") & (F.col("faixa_renda") != "")).groupBy("faixa_renda", "ano").agg(
    F.mean("media_objetiva").alias("avg_objetiva")
)
low = avg_renda_ano.filter(F.col("faixa_renda") == "A").withColumnRenamed("avg_objetiva", "low").select("ano", "low")
high = avg_renda_ano.filter(F.col("faixa_renda") == "Q").withColumnRenamed("avg_objetiva", "high").select("ano", "high")
gap_df = low.join(high, "ano").withColumn("gap", F.col("high") - F.col("low")).orderBy("ano")
rows_gap = gap_df.collect()
years_v5 = [r.ano for r in rows_gap]
gaps = [round(r.gap, 1) for r in rows_gap]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(years_v5, gaps, marker="o", linewidth=2, markersize=8)
ax.set_xlabel("Year")
ax.set_ylabel("Score gap (highest income Q - lowest income A)")
ax.set_title("Performance Gap by Income (Q006: Highest vs Lowest Bracket) Over Time")
ax.set_ylim(0, None)
plt.tight_layout()
plt.savefig(FIG_DIR / "v5_income_gap_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gap (high-low) by year:", dict(zip(years_v5, gaps)))

## Visual 6 — Performance by Gender Over Years

In [ ]:
by_sex_ano = f_perfil.filter(F.col("tp_sexo").isin("M", "F")).groupBy("tp_sexo", "ano").agg(
    F.mean("media_objetiva").alias("avg_objetiva")
).orderBy("ano", "tp_sexo")
rows_v6 = by_sex_ano.collect()
m_ano = {r.ano: r.avg_objetiva for r in rows_v6 if r.tp_sexo == "M"}
f_ano = {r.ano: r.avg_objetiva for r in rows_v6 if r.tp_sexo == "F"}
years_v6 = sorted(set(m_ano) | set(f_ano))
m_vals = [m_ano.get(y) for y in years_v6]
f_vals = [f_ano.get(y) for y in years_v6]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_v6, m_vals, marker="o", label="M", linewidth=2)
ax.plot(years_v6, f_vals, marker="s", label="F", linewidth=2)
ax.set_xlabel("Year")
ax.set_ylabel("Average objective mean score")
ax.set_title("Performance by Gender (Mean Objective Score) Over Years")
ax.set_ylim(0, 1000)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "v6_performance_by_gender.png", dpi=150, bbox_inches="tight")
plt.show()
print("M:", m_ano, "F:", f_ano)

## Statistical Summary per Visual

In [ ]:
print("=" * 60)
print("V1 — Media objetiva: Same scale 0–1000; distributions comparable by year.")
print("V2 — Redação: Tail >= 800 and % above 800 by year; comparable scale.")
print("V3 — Presence: Gold valid cohort shows 100% fully present; pandemic band 2020–2021.")
print("V4 — UF growth: Top 10 UFs by delta (2024−2020) in average score; same y-axis 0–1000.")
print("V5 — Income gap: Gap = mean(Q)−mean(A) over years; equity trend.")
print("V6 — Gender: Mean objective score M vs F 2020–2024; same scale 0–1000.")
print("=" * 60)
spark.stop()